In [40]:
import os
from dotenv import load_dotenv
from openai import OpenAI

import gradio as gr

In [42]:
load_dotenv(override=True);
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")


Google API Key exists and begins AIzaSyBp
Groq API Key exists and begins gsk_4D0P


In [68]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openai = OpenAI(base_url=ollama_url, api_key='ollama')

In [44]:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short and simple description about the company model for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [45]:
from bs4 import BeautifulSoup
import requests

# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [69]:

def stream_gemini(prompt):
    """
    Stream the response from the Gemini model for the given prompt.
    """
    stream = gemini.chat.completions.create(
        model='gemini-2.5-flash',
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [70]:
def stream_groq(prompt):
    """
    Stream the response from the Groq model for the given prompt.
    """
    stream = groq.chat.completions.create(
        model='openai/gpt-oss-120b',
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [71]:
def stream_ollama(prompt):
    """
    Stream the response from the Ollama model for the given prompt.
    """
    stream = openai.chat.completions.create(
        model='llama3.2:1b',
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [84]:
def stream_broucher(model,url,customUrl):
    """
    Stream the brochure content for the given URL and model.
    """
    yield ""
    prompt = f"Please generate a company brochure for {url}. Here is their landing page:\n"
    website = customUrl.strip() if customUrl else url
    prompt = fetch_website_contents(website)
    if model == "GEMINI":
        result = stream_gemini(prompt)
    elif model == "GROQ":
        result = stream_groq(prompt)
    elif model == "OLLAMA":
        result = stream_ollama(prompt)
    else:
        raise ValueError(f"Unknown model: {model}")
    yield from result

In [85]:
website_collection = ["https://www.figma.com/", "https://www.microsoft.com", "https://www.apple.com", "https://www.amazon.com", "https://github.com/", "https://www.twitter.com", "https://www.linkedin.com", "https://www.netflix.com", "https://www.spotify.com"]

In [86]:
model_selector = gr.Dropdown([ "GEMINI", "OLLAMA","GROQ"], label="Select model", value="GEMINI")
website_selector = gr.Dropdown(website_collection, label="Select website", value=website_collection[0])
custom_website = gr.Textbox(
    label="Or Enter Custom Website URL",
    placeholder="https://example.com"
)

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_broucher,
    title="Business Model Overview", 
    inputs=[ model_selector, website_selector, custom_website], 
    outputs=[message_output], 
    flagging_mode="never"
    ).launch(inbrowser=True,auth=[("Test", "Test@123")])

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "d:\AI_Projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 849, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_Projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_Projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_Projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AI_Projects\llm_engineering\.venv\Lib\site-packages\gradio\utils.py", line 760, in async_iteration
    return await ane